# 02 — Feature Engineering
### FUT 26 Player Price Dataset

In this notebook, we transform the raw scraped data into a feature‑rich dataset suitable for machine learning.

We will create:
- lag features  
- moving averages  
- volatility indicators  
- calendar features  
- the target variable (next day price)  

These features will be used in Notebook 3 for model training.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append('..')
from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.features import build_features

## 1. Load Raw Data

We load the raw CSV created by the scraper.

In [ ]:
df_raw = pd.read_csv(RAW_DATA_DIR/"fut_prices_raw.csv")
df_raw["date"] = pd.to_datetime(df_raw["date"])
df_raw.head()

,player_id,player_name,date,price
0,254,lionel-messi,2025-09-18,33512
1,254,lionel-messi,2025-09-19,31182
2,254,lionel-messi,2025-09-20,23920
3,254,lionel-messi,2025-09-21,29705
4,254,lionel-messi,2025-09-22,35307


## 2. Sort and Inspect

We sort by player and date to ensure correct time‑series ordering.

In [3]:
df_raw = df_raw.sort_values(["player_name", "date"])
df_raw.head()

,player_id,player_name,date,price
688,1009,alphonso-davies,2025-09-18,33638
689,1009,alphonso-davies,2025-09-19,32234
690,1009,alphonso-davies,2025-09-20,25099
691,1009,alphonso-davies,2025-09-21,25932
692,1009,alphonso-davies,2025-09-22,27427


## 3. Apply Feature Engineering

We use the `build_features()` function from `features.py`.

This function creates:
- lag features (1, 3, 7 days)
- moving averages (3, 7 days)
- volatility indicators (3, 7 days)
- day of week
- weekend flag
- target variable: next_day_price

In [4]:
df_features = build_features(df_raw)
df_features.head()

,player_id,player_name,date,price,log_price,lag_1,lag_3,lag_7,ma_3,ma_7,pct_change,vol_3,vol_7,day_of_week,is_weekend,next_day_price
351,41,florian-wirtz,2025-09-25,296125,12.598540,12.590205,12.622682,12.495584,12.607686,12.616885,0.000662,0.002476,0.007733,3,0,12.573583
352,41,florian-wirtz,2025-09-26,288826,12.573583,12.598540,12.634313,12.701025,12.587443,12.598679,-0.001981,0.002102,0.003831,4,0,12.659493
353,41,florian-wirtz,2025-09-27,314736,12.659493,12.573583,12.590205,12.600204,12.610539,12.607149,0.006833,0.004523,0.003718,5,1,12.633723
354,41,florian-wirtz,2025-09-28,306729,12.633723,12.659493,12.598540,12.571226,12.622266,12.616077,-0.002036,0.005104,0.003683,6,1,12.630618
355,41,florian-wirtz,2025-09-29,305778,12.630618,12.633723,12.573583,12.622682,12.641278,12.617211,-0.000246,0.004690,0.003372,0,0,12.619777


## 4. Check Missing Values

After feature engineering, early rows will be dropped due to lag/rolling windows.

In [5]:
df_features.isna().sum()

player_id         0
player_name       0
date              0
price             0
log_price         0
lag_1             0
lag_3             0
lag_7             0
ma_3              0
ma_7              0
pct_change        0
vol_3             0
vol_7             0
day_of_week       0
is_weekend        0
next_day_price    0
dtype: int64

## 5. Feature List

Let’s inspect all engineered features.

In [6]:
df_features.columns

Index(['player_id', 'player_name', 'date', 'price', 'log_price', 'lag_1',
       'lag_3', 'lag_7', 'ma_3', 'ma_7', 'pct_change', 'vol_3', 'vol_7',
       'day_of_week', 'is_weekend', 'next_day_price'],
      dtype='object')

## 6. Visualizing Lag Features

Lag features help the model understand short‑term momentum.

In [14]:
import plotly.express as px

sample_player = df_features["player_name"].unique()[0]
player_data = df_features[df_features["player_name"] == sample_player]

fig = px.line(player_data, x="date", y=["log_price", "lag_1", "lag_3", "lag_7"], title=f"Lag Features — {sample_player}")
fig.update_xaxes(title_text="Date", tickangle=45)
fig.update_yaxes(title_text="Value")
fig.show()

## 7. Visualizing Moving Averages

Moving averages smooth out noise and capture trend direction.

In [13]:
sample_player = df_features["player_name"].unique()[0]
player_data = df_features[df_features["player_name"] == sample_player]

fig = px.line(player_data, x="date", y=["log_price", "ma_3", "ma_7"], title=f"Moving Averages — {sample_player}")
fig.update_xaxes(title_text="Date", tickangle=45)
fig.update_yaxes(title_text="Price")
fig.show()

## 8. Visualizing Volatility

Volatility features help the model understand how “risky” or unstable a player’s price is.

In [9]:
sample_player = df_features["player_name"].unique()[0]
player_data = df_features[df_features["player_name"] == sample_player]

fig = px.line(player_data, x="date", y=["vol_3", "vol_7"], title=f"Volatility Indicators — {sample_player}")
fig.update_xaxes(title_text="Date", tickangle=45)
fig.update_yaxes(title_text="Volatility")
fig.show()

## 9. Calendar Features

Day‑of‑week and weekend flags capture weekly market behavior:
- Thursday rises (rewards)
- Weekend dips (WL sell‑off)

In [10]:
df_features[["day_of_week", "is_weekend"]].head()

,day_of_week,is_weekend
351,3,0
352,4,0
353,5,1
354,6,1
355,0,0


## 10. Target Variable

The target is **next_day_price**, created using:

df["next_day_price"] = df["price"].shift(-1)


This aligns tomorrow’s price with today’s features.

In [15]:
df_features[["log_price", "next_day_price"]].head(10)

,log_price,next_day_price
351,12.598540,12.573583
352,12.573583,12.659493
353,12.659493,12.633723
354,12.633723,12.630618
355,12.630618,12.619777
356,12.619777,12.560700
357,12.560700,12.601982
358,12.601982,12.635245
359,12.635245,12.584917
360,12.584917,12.556170


## 11. Save Processed Dataset

We save the processed dataset for Notebook 3 (modeling).

In [12]:
df_features.to_csv(f"../{PROCESSED_DATA_DIR}/fut_prices_features.csv", index=False)

# 12. Conclusions

We successfully engineered a rich set of features:

### ✔ Lag features  
Capture short‑term momentum.

### ✔ Moving averages  
Capture trend direction.

### ✔ Volatility indicators  
Capture price stability and risk.

### ✔ Calendar features  
Capture weekly market behavior.

### ✔ Target variable  
Aligns tomorrow’s price with today’s features.

This dataset is now ready for model training in **Notebook 3 — Modeling**.